# 🔄 Notebook 2: Simple Polling

Simple polling is the most straightforward approach to getting updates from a server. It's not truly "real-time," but it's a great baseline and works for many use cases!

## Learning Objectives

By the end of this notebook, you'll understand:
- How simple polling works
- When to use it (and when not to)
- How to implement a polling client and server
- Trade-offs and optimizations

## 🤔 What is Simple Polling?

Simple polling is exactly what it sounds like: the client repeatedly asks the server "Do you have anything new for me?"

```
┌────────┐                              ┌────────┐
│ Client │                              │ Server │
└───┬────┘                              └───┬────┘
    │                                       │
    │──── GET /updates ────────────────────►│
    │◄─── {"updates": []} ─────────────────│  (nothing new)
    │                                       │
    │     ⏳ Wait 2 seconds...              │
    │                                       │
    │──── GET /updates ────────────────────►│
    │◄─── {"updates": []} ─────────────────│  (still nothing)
    │                                       │
    │     ⏳ Wait 2 seconds...              │
    │                                       │
    │──── GET /updates ────────────────────►│
    │◄─── {"updates": [{...}]} ────────────│  (got something!)
    │                                       │
```

It's like checking your mailbox every hour. Simple, but you might miss time-sensitive letters!

## 🛠️ Let's Build It!

We'll create a simple chat room where messages are fetched via polling.

### Step 1: Start the Server

The next cell starts `servers/simple_polling_server.py` for you and shuts it
down when the kernel exits, so there is nothing to do by hand.

If you would rather watch the server's log live, start it yourself in a second
terminal first — the notebook detects the already-listening port and leaves
your process alone:

```bash
cd 04-patterns/real-time-updates/servers
python simple_polling_server.py     # 🚀 Starting Simple Polling Server on port 5001
```

In [ ]:
# Start the simple polling server this notebook talks to.
#
# `ensure_server` is idempotent: if you already started the server by hand
# in another terminal it is left alone, otherwise it is launched as a
# background process using this notebook's own interpreter (the lab .venv)
# and shut down when the kernel exits. This is what makes the notebook
# runnable on its own -- previously the next cell just died with a raw
# ConnectionError if you had not started the server first.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "servers"))
from lab_servers import ensure_server

print(ensure_server(5001))

import requests

health = requests.get("http://localhost:5001/health", timeout=5)
health.raise_for_status()
print("health:", health.json())


### Step 2: Create the Polling Client

In [ ]:
import requests
import time
from datetime import datetime

class SimplePollingClient:
    """
    A simple polling client that checks for new messages at regular intervals.
    """
    
    def __init__(self, server_url: str, poll_interval: float = 2.0):
        self.server_url = server_url
        self.poll_interval = poll_interval
        self.last_timestamp = 0  # Track when we last got messages
        self.running = False
    
    def poll_once(self):
        """
        Make a single poll request to the server.
        Returns new messages since last poll.
        """
        try:
            response = requests.get(
                f"{self.server_url}/messages",
                params={"since": self.last_timestamp},
                timeout=5
            )
            
            if response.status_code == 200:
                data = response.json()
                messages = data.get("messages", [])
                
                # Update our timestamp to the latest message
                if messages:
                    self.last_timestamp = max(msg["timestamp"] for msg in messages)
                
                return messages
            else:
                print(f"⚠️ Server returned status {response.status_code}")
                return []
                
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed: {e}")
            return []
    
    def send_message(self, user: str, text: str):
        """
        Send a message to the chat.
        """
        try:
            response = requests.post(
                f"{self.server_url}/messages",
                json={"user": user, "text": text},
                timeout=5
            )
            return response.status_code == 201
        except requests.exceptions.RequestException as e:
            print(f"❌ Failed to send message: {e}")
            return False

# Create client instance
client = SimplePollingClient("http://localhost:5001")
print("✅ Polling client created!")

In [ ]:
# Let's send some test messages!

print("📤 Sending test messages...\n")

client.send_message("Alice", "Hello everyone!")
time.sleep(0.5)
client.send_message("Bob", "Hey Alice! How are you?")
time.sleep(0.5)
client.send_message("Alice", "Doing great! Just learning about polling.")

print("\n✅ Messages sent!")

In [ ]:
# Now let's poll for messages!

print("🔄 Polling for messages...\n")

messages = client.poll_once()

if messages:
    print(f"📬 Received {len(messages)} message(s):\n")
    for msg in messages:
        timestamp = datetime.fromtimestamp(msg['timestamp']).strftime('%H:%M:%S')
        print(f"  [{timestamp}] {msg['user']}: {msg['text']}")
else:
    print("📭 No new messages")

In [ ]:
# Let's see continuous polling in action!
# We'll poll for 10 seconds and see what happens

import threading

def continuous_poll(client, duration=10, interval=2):
    """
    Poll continuously for a set duration.
    """
    print(f"🔄 Starting continuous polling for {duration} seconds...")
    print(f"   Polling every {interval} seconds\n")
    
    start_time = time.time()
    poll_count = 0
    
    while time.time() - start_time < duration:
        poll_count += 1
        current_time = datetime.now().strftime('%H:%M:%S')
        
        messages = client.poll_once()
        
        if messages:
            print(f"[{current_time}] Poll #{poll_count}: 📬 {len(messages)} new message(s)")
            for msg in messages:
                print(f"            └─ {msg['user']}: {msg['text']}")
        else:
            print(f"[{current_time}] Poll #{poll_count}: 📭 No new messages")
        
        time.sleep(interval)
    
    print(f"\n✅ Polling complete! Made {poll_count} requests.")

# Reset client timestamp to see all messages
client.last_timestamp = 0

# Start polling in background
poll_thread = threading.Thread(target=continuous_poll, args=(client, 10, 2))
poll_thread.start()

# Wait a bit then send a new message
time.sleep(5)
print("\n📤 Sending a new message during polling...\n")
client.send_message("Charlie", "I just joined!")

# Wait for polling to complete
poll_thread.join()

## 📊 Understanding the Trade-offs

Let's visualize what happens with different polling intervals:

In [ ]:
def analyze_polling_tradeoffs():
    """
    Analyze the trade-offs of different polling intervals.
    """
    print("📊 Polling Interval Trade-offs")
    print("="*60)
    print("")
    
    # Calculate requests per hour for different intervals
    intervals = [
        (0.5, "Very fast (500ms)"),
        (2, "Fast (2s)"),
        (5, "Medium (5s)"),
        (30, "Slow (30s)"),
        (60, "Very slow (1min)")
    ]
    
    print(f"{'Interval':<25} {'Requests/Hour':<15} {'Max Latency':<15}")
    print("-"*55)
    
    for interval, name in intervals:
        requests_per_hour = 3600 / interval
        max_latency = f"{interval}s"
        print(f"{name:<25} {requests_per_hour:<15.0f} {max_latency:<15}")
    
    print("")
    print("💡 Key Insight:")
    print("   - Faster polling = More requests = Higher server load")
    print("   - Slower polling = Fewer requests = Higher latency")
    print("")
    print("📈 With 10,000 users:")
    print("   - 500ms polling: 72 million requests/hour!")
    print("   - 30s polling: 1.2 million requests/hour")

analyze_polling_tradeoffs()

## ✅ Advantages of Simple Polling

1. **Dead simple to implement** - Just HTTP requests!
2. **Stateless** - No connection to maintain
3. **Works everywhere** - Any HTTP client works
4. **Easy to debug** - Standard request/response
5. **No special infrastructure** - Works with any load balancer

## ❌ Disadvantages

1. **Higher latency** - Updates delayed by polling interval
2. **Wasted requests** - Most polls return empty
3. **Inefficient** - Server load scales with users × poll rate
4. **Not truly real-time** - Inherent delay

In [ ]:
# Let's measure the "wasted" requests problem.
#
# The point only lands if messages ACTUALLY arrive during the run -- a loop
# that polls an idle server is guaranteed to report 0% useful and proves
# nothing. So we publish one message from a background thread, part-way in,
# and measure the real hit rate.

import threading

def measure_polling_efficiency(poll_count=10, interval=1, send_at_poll=6):
    """Poll `poll_count` times; one message is published before poll #6."""
    client.last_timestamp = time.time()

    def publish_once():
        # Land squarely inside the window for poll #`send_at_poll`.
        time.sleep(interval * (send_at_poll - 1) - interval / 2)
        client.send_message("Dana", "the one message in this whole window")

    publisher = threading.Thread(target=publish_once)
    publisher.start()

    empty_polls = 0
    useful_polls = 0

    print(f"🔬 Running {poll_count} polls with {interval}s interval...")
    print(f"   (exactly ONE message will be published, before poll #{send_at_poll})\n")

    for i in range(poll_count):
        messages = client.poll_once()
        if messages:
            useful_polls += 1
            print(f"  Poll {i+1}: 📬 Got {len(messages)} message(s)")
        else:
            empty_polls += 1
            print(f"  Poll {i+1}: 📭 Empty")
        time.sleep(interval)

    publisher.join()

    efficiency = (useful_polls / poll_count) * 100

    print(f"\n📊 Results:")
    print(f"   Empty polls:  {empty_polls} ({100-efficiency:.1f}%)")
    print(f"   Useful polls: {useful_polls} ({efficiency:.1f}%)")
    print(f"\n💡 One event, {poll_count} requests. Everything except {useful_polls} of")
    print(f"   them was pure overhead -- and that ratio gets WORSE the faster you poll.")

    # Fail loudly if this stops demonstrating its own lesson: the message must
    # actually be delivered (otherwise we are back to the degenerate version),
    # and the vast majority of polls must still come back empty.
    assert useful_polls >= 1, "the published message was never delivered -- demo is broken"
    assert empty_polls >= poll_count - 2, (
        f"expected almost every poll to be wasted, got {empty_polls}/{poll_count} empty"
    )
    return efficiency

measure_polling_efficiency()

## 🎯 When to Use Simple Polling

Simple polling is a great choice when:

| Use Case | Why Polling Works |
|----------|-------------------|
| Dashboard updates | 5-10s delay is acceptable |
| Email inbox | Users don't expect instant updates |
| Social media feeds | "Pull to refresh" is expected |
| Status pages | Updates are infrequent |
| Analytics | Near real-time is good enough |

### Don't use it when:

- Users expect **instant** updates (chat, gaming)
- Updates happen **very frequently** (stock tickers)
- You have **millions of users** (server load issue)
- Latency is **critical** (live auctions)

## 🔧 Optimization: HTTP Keep-Alive

One way to reduce polling overhead is using HTTP keep-alive connections. This avoids the TCP handshake for each request.

In [ ]:
import statistics
import time

import requests

def compare_keep_alive(samples: int = 40):
    """
    Compare request times with and without keep-alive.

    Loopback is a hostile place to measure this: the TCP handshake to
    127.0.0.1 costs microseconds, so we need enough samples (and medians
    rather than means) before the difference rises above the noise. We print
    the measured verdict either way -- including "too close to call".
    """
    url = "http://localhost:5001/messages?since=0"

    def timed(get):
        t0 = time.perf_counter()
        get(url, timeout=5)
        return (time.perf_counter() - t0) * 1000

    # Without keep-alive: requests.get() builds and tears down a Session
    # (and therefore a TCP connection) on every single call.
    times_no_ka = [timed(requests.get) for _ in range(samples)]

    # With keep-alive: one Session, one connection, reused. Skip the first
    # request -- it is the one that still pays for the handshake.
    session = requests.Session()
    timed(session.get)
    times_ka = [timed(session.get) for _ in range(samples)]
    session.close()

    med_no_ka = statistics.median(times_no_ka)
    med_ka = statistics.median(times_ka)

    print(f"🔄 Without keep-alive ({samples} requests): median {med_no_ka:.3f}ms")
    print(f"🔗 With keep-alive    ({samples} requests): median {med_ka:.3f}ms")

    saved = med_no_ka - med_ka
    if saved > 0:
        print(f"\n✅ Keep-alive saved {saved:.3f}ms/request ({saved/med_no_ka:.0%}).")
    else:
        print(f"\n⚖️  Too close to call on loopback ({saved:+.3f}ms).")

    print("\n💡 What you just measured on loopback is socket setup + teardown, which")
    print("   is already worth a large fraction of a request. Over a real network the")
    print("   same saving is a whole round trip (often 30-100ms), and 2-3 more on top")
    print("   for the TLS handshake. Every polling client you ship should reuse a")
    print("   connection -- this is the cheapest optimisation in the whole notebook.")
    return med_no_ka, med_ka

compare_keep_alive()

## 🧪 Quick Quiz

1. **You're building a weather dashboard that updates every 5 minutes. Should you use simple polling?**

2. **Your polling interval is 2 seconds. What's the maximum delay a user might experience for seeing a new message?**

3. **You have 100,000 users polling every 5 seconds. How many requests per second is that?**

In [ ]:
# Run this to see the answers!

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. YES! Weather updates every 5 minutes is perfect for polling.")
print("   You could poll every 1-2 minutes and still be very responsive.")
print("")
print("2. Maximum delay = 2 seconds (the polling interval) + network latency")
print("   If a message arrives right after a poll, user waits the full interval.")
print("")
print("3. 100,000 users ÷ 5 seconds = 20,000 requests per second!")
print("   This is why polling doesn't scale well for large user bases.")

## 📚 Summary

### What We Learned:

1. **Simple polling** = Client repeatedly asks server for updates
2. **Easy to implement** but not truly real-time
3. **Trade-off**: Faster polling = more load, slower = more latency
4. **Most polls are wasted** - return no new data
5. **HTTP keep-alive** reduces overhead

### Interview Tips:

> "I'm going to start with a simple polling approach so I can focus on [core problem]. We can switch to something more sophisticated if we need lower latency."

This shows you understand trade-offs and can prioritize!

### Next Up: Long Polling

In the next notebook, we'll see how **long polling** improves on simple polling by holding requests open until there's new data.

In [ ]:
# Cleanup
#
# `ensure_server` registered an atexit hook, so any server IT started is shut
# down automatically when this kernel exits. Call stop_all() if you want the
# port free right now. A server you started yourself in a terminal is never
# touched -- stop that one with Ctrl+C.
from lab_servers import stop_all, is_listening

stop_all()
print("🧹 Cleanup:")
print(f"   port 5001 still listening? {is_listening(5001)}")
print("   (True means you started the server yourself -- Ctrl+C in that terminal.)")